# Exp 23: Structural KC Analysis — Full-rules vs No-rules V3

Tests whether removing disambiguation rules causes the V3 model to tag the five
structural KCs (If/Else, NestedIf, While, For, NestedFor) more often, and whether
that raises agreement with the human raters on those KCs.

**Data**: 10 students rated under V3 rules by both humans and the ablation runs.  
**Sources**: Human A (Pranay), Human B (Arundhati), Full-rules V3 (baseline ablation), No-rules V3.

In [1]:
import json
import os
import sys
from pathlib import Path
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'utils').exists() and (ROOT.parent / 'utils').exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import KC_COLUMNS, problem_f1, compute_kappa

print(f'Working directory: {ROOT}')
print(f'KC_COLUMNS ({len(KC_COLUMNS)}): {KC_COLUMNS}')

Working directory: d:\Projects\kintsugi
KC_COLUMNS (18): ['If/Else', 'NestedIf', 'While', 'For', 'NestedFor', 'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean', 'StringFormat', 'StringConcat', 'StringIndex', 'StringLen', 'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction']


In [2]:
# Annotation loading — same pattern as exp12_v3_prompt_experiment.ipynb

def load_annotations(filepath: str) -> Tuple[str, Dict[str, Set[str]]]:
    path = Path(filepath)
    if not path.exists():
        print(f'  WARNING: not found: {path}')
        return path.stem, {}
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    rater_name = data.get('rater', path.stem)
    student_id = str(data.get('student_id', data.get('studentId', 'unknown')))
    parsed: Dict[str, Set[str]] = {}
    for pid, val in data.get('annotations', {}).items():
        comp_pid = f'{student_id}_{pid}'
        if isinstance(val, dict) and 'gaps' in val:
            gaps = val['gaps']
            parsed[comp_pid] = set(gaps) if isinstance(gaps, list) else set()
        elif isinstance(val, list):
            parsed[comp_pid] = set(val)
        else:
            parsed[comp_pid] = set()
        parsed[comp_pid] = {tag for tag in parsed[comp_pid] if tag in KC_COLUMNS}
    return rater_name, parsed


def merge_rater_files(filepaths: List[str], label: str) -> Tuple[str, Dict[str, Set[str]]]:
    merged: Dict[str, Set[str]] = {}
    for fp in filepaths:
        _, anns = load_annotations(fp)
        for comp_pid, kcs in anns.items():
            if comp_pid in merged:
                merged[comp_pid].update(kcs)
            else:
                merged[comp_pid] = set(kcs)
    return label, merged

In [3]:
# File paths — 10 ablation students rated with V3 rules by both humans
STUDENTS = [10155, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499, 9948]

RATED_DIR = Path('dataset/Rater_KC_Tags/Rated_KC_V3')
ABLATION_DIR = Path('results/human_validation/ablation')

HA_FILES   = sorted(str(p) for p in RATED_DIR.glob('kc_annotations_Pranay Ghuge_*.json'))
HB_FILES   = sorted(str(p) for p in RATED_DIR.glob('kc_annotations_Arundhati Das_*.json'))
FULL_FILES = [str(ABLATION_DIR / f'llm_ablation_baseline_{sid}.json') for sid in STUDENTS]
NR_FILES   = [str(ABLATION_DIR / f'llm_ablation_no_rules_{sid}.json') for sid in STUDENTS]

print(f'Human A files  ({len(HA_FILES)}): {[Path(f).name for f in HA_FILES]}')
print(f'Human B files  ({len(HB_FILES)}): {[Path(f).name for f in HB_FILES]}')
print(f'Full-rules files ({len(FULL_FILES)})')
print(f'No-rules files   ({len(NR_FILES)})')

Human A files  (10): ['kc_annotations_Pranay Ghuge_10155_1778019927566.json', 'kc_annotations_Pranay Ghuge_14189_1776827078260.json', 'kc_annotations_Pranay Ghuge_14352_1777493137819.json', 'kc_annotations_Pranay Ghuge_14362_1777510916068.json', 'kc_annotations_Pranay Ghuge_14363_1778009820596.json', 'kc_annotations_Pranay Ghuge_14374_1778011754339.json', 'kc_annotations_Pranay Ghuge_14414_1778013558136.json', 'kc_annotations_Pranay Ghuge_14474_1778019006374.json', 'kc_annotations_Pranay Ghuge_14499_1778014166331.json', 'kc_annotations_Pranay Ghuge_9948_1776814758492.json']
Human B files  (10): ['kc_annotations_Arundhati Das_10155_1777491837576.json', 'kc_annotations_Arundhati Das_14189_1778021530364.json', 'kc_annotations_Arundhati Das_14352_1777911317543.json', 'kc_annotations_Arundhati Das_14362_1777912347301.json', 'kc_annotations_Arundhati Das_14363_1777913170205.json', 'kc_annotations_Arundhati Das_14374_1778011329291.json', 'kc_annotations_Arundhati Das_14414_1778012668437.json'

In [4]:
# Load all four sources and intersect to common pairs

_, anns_ha   = merge_rater_files(HA_FILES,   'Human A')
_, anns_hb   = merge_rater_files(HB_FILES,   'Human B')
_, anns_full = merge_rater_files(FULL_FILES,  'Full-rules V3')
_, anns_nr   = merge_rater_files(NR_FILES,    'No-rules V3')

common_pids = sorted(
    set(anns_ha) & set(anns_hb) & set(anns_full) & set(anns_nr)
)
print(f'Common (student, problem) pairs across all 4 sources: {len(common_pids)}')
print(f'  Human A  total pairs: {len(anns_ha)}')
print(f'  Human B  total pairs: {len(anns_hb)}')
print(f'  Full-rules total pairs: {len(anns_full)}')
print(f'  No-rules  total pairs: {len(anns_nr)}')

Common (student, problem) pairs across all 4 sources: 372
  Human A  total pairs: 372
  Human B  total pairs: 372
  Full-rules total pairs: 372
  No-rules  total pairs: 372


In [5]:
# Per-KC metric helpers
#
# kc_f1_vs_human: treats the set of pairs that tagged KC as a set, applies
# the same precision/recall definition as problem_f1 (set_a=reference,
# set_b=predicted). Both-empty = 1.0 per problem_f1 contract.
#
# kc_kappa_vs_human: builds 0/1 decision lists across all pairs, calls
# compute_kappa from utils.metrics.

def kc_f1_vs_human(
    anns_model: Dict[str, Set[str]],
    anns_human: Dict[str, Set[str]],
    pids: List[str],
    kc: str,
) -> float:
    set_h = {pid for pid in pids if kc in anns_human.get(pid, set())}
    set_m = {pid for pid in pids if kc in anns_model.get(pid, set())}
    return problem_f1(set_h, set_m)


def kc_kappa_vs_human(
    anns_model: Dict[str, Set[str]],
    anns_human: Dict[str, Set[str]],
    pids: List[str],
    kc: str,
) -> float:
    bh = [1 if kc in anns_human.get(pid, set()) else 0 for pid in pids]
    bm = [1 if kc in anns_model.get(pid, set()) else 0 for pid in pids]
    try:
        return compute_kappa(bh, bm)
    except Exception:
        return float('nan')

In [6]:
# Compute per-KC table
STRUCTURAL = ['If/Else', 'NestedIf', 'While', 'For', 'NestedFor']

rows = []
for kc in KC_COLUMNS:
    ha_count   = sum(1 for p in common_pids if kc in anns_ha.get(p, set()))
    hb_count   = sum(1 for p in common_pids if kc in anns_hb.get(p, set()))
    full_count = sum(1 for p in common_pids if kc in anns_full.get(p, set()))
    nr_count   = sum(1 for p in common_pids if kc in anns_nr.get(p, set()))

    f1_full = (kc_f1_vs_human(anns_full, anns_ha, common_pids, kc) +
               kc_f1_vs_human(anns_full, anns_hb, common_pids, kc)) / 2
    f1_nr   = (kc_f1_vs_human(anns_nr,   anns_ha, common_pids, kc) +
               kc_f1_vs_human(anns_nr,   anns_hb, common_pids, kc)) / 2

    k_full  = (kc_kappa_vs_human(anns_full, anns_ha, common_pids, kc) +
               kc_kappa_vs_human(anns_full, anns_hb, common_pids, kc)) / 2
    k_nr    = (kc_kappa_vs_human(anns_nr,   anns_ha, common_pids, kc) +
               kc_kappa_vs_human(anns_nr,   anns_hb, common_pids, kc)) / 2

    rows.append({
        'KC':           kc,
        'HumA_count':   ha_count,
        'HumB_count':   hb_count,
        'Full_count':   full_count,
        'NoRules_count': nr_count,
        'F1_full':      round(f1_full, 4),
        'F1_norules':   round(f1_nr,   4),
        'F1_delta':     round(f1_nr - f1_full, 4),
        'Kappa_full':   round(k_full,  4),
        'Kappa_norules': round(k_nr,   4),
        'Kappa_delta':  round(k_nr - k_full, 4),
    })

# Structural first, then the rest
struct_rows = [r for r in rows if r['KC'] in STRUCTURAL]
other_rows  = [r for r in rows if r['KC'] not in STRUCTURAL]

# Aggregate row for the 5 structural KCs combined
agg = {
    'KC':            'STRUCTURAL_COMBINED',
    'HumA_count':    sum(r['HumA_count']   for r in struct_rows),
    'HumB_count':    sum(r['HumB_count']   for r in struct_rows),
    'Full_count':    sum(r['Full_count']   for r in struct_rows),
    'NoRules_count': sum(r['NoRules_count'] for r in struct_rows),
    'F1_full':       round(np.mean([r['F1_full']      for r in struct_rows]), 4),
    'F1_norules':    round(np.mean([r['F1_norules']   for r in struct_rows]), 4),
    'F1_delta':      round(np.mean([r['F1_delta']     for r in struct_rows]), 4),
    'Kappa_full':    round(np.nanmean([r['Kappa_full']     for r in struct_rows]), 4),
    'Kappa_norules': round(np.nanmean([r['Kappa_norules']  for r in struct_rows]), 4),
    'Kappa_delta':   round(np.nanmean([r['Kappa_delta']    for r in struct_rows]), 4),
}

all_rows = struct_rows + other_rows + [agg]
df = pd.DataFrame(all_rows)
print(df.to_string(index=False))

                 KC  HumA_count  HumB_count  Full_count  NoRules_count  F1_full  F1_norules  F1_delta  Kappa_full  Kappa_norules  Kappa_delta
            If/Else          17           8          37             35   0.3333      0.3318   -0.0015      0.3005         0.2986      -0.0019
           NestedIf           0           1           0              1   0.5000      0.0000   -0.5000         NaN        -0.0013          NaN
              While           3           2           5              5   0.6607      0.6607    0.0000      0.6578         0.6578       0.0000
                For          23          10          21             21   0.5440      0.5762    0.0323      0.5222         0.5557       0.0335
          NestedFor           1           0           1              1   0.0000      0.0000    0.0000     -0.0013        -0.0013       0.0000
           Math+-*/          16          19          19             20   0.5774      0.5855    0.0080      0.5556         0.5637       0.0081
      

d:\Projects\kintsugi\.venv\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
d:\Projects\kintsugi\.venv\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedMetricWarning: `y1`, `y2` and `labels` have only one label in common. `cohen_kappa_score` is undefined and set to the value defined by the the `replace_undefined_by` param, which is set to nan.
  return func(*args, **kwargs)


In [7]:
# Save CSV
OUT_CSV = Path('results/structural_kc_analysis.csv')
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f'Saved {OUT_CSV}')

Saved results\structural_kc_analysis.csv


In [8]:
# Figure 1: Grouped bar chart — structural KC tag counts across 4 sources
FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

struct_df = df[df['KC'].isin(STRUCTURAL)].set_index('KC').reindex(STRUCTURAL)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(STRUCTURAL))
w = 0.2

colors = ['#4477AA', '#66CCEE', '#228833', '#CCBB44']
ax.bar(x - 1.5*w, struct_df['HumA_count'],   width=w, label='Human A',      color=colors[0])
ax.bar(x - 0.5*w, struct_df['HumB_count'],   width=w, label='Human B',      color=colors[1])
ax.bar(x + 0.5*w, struct_df['Full_count'],   width=w, label='Full-rules V3', color=colors[2])
ax.bar(x + 1.5*w, struct_df['NoRules_count'], width=w, label='No-rules V3', color=colors[3])

ax.set_xticks(x)
ax.set_xticklabels(STRUCTURAL, fontsize=11)
ax.set_ylabel('Tag count (pairs)', fontsize=11)
ax.set_title(
    'Structural KC tag counts: do no-rules model counts move toward human counts?',
    fontsize=11,
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig1_path = FIG_DIR / 'fig1_structural_kc_counts.png'
fig.savefig(fig1_path, dpi=300)
plt.close(fig)
print(f'Saved {fig1_path}')

Saved results\figures\fig1_structural_kc_counts.png


In [9]:
# Figure 2: Diverging bar chart — F1_delta (no-rules minus full) for all 18 KCs
kc18_df = df[df['KC'] != 'STRUCTURAL_COMBINED'].copy()
kc18_df = kc18_df.sort_values('F1_delta', ascending=True).reset_index(drop=True)

is_structural = kc18_df['KC'].isin(STRUCTURAL).values
bar_colors = ['#EE6677' if s else '#BBBBBB' for s in is_structural]

fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(kc18_df))
bars = ax.barh(y, kc18_df['F1_delta'], color=bar_colors, edgecolor='white', linewidth=0.5)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(kc18_df['KC'], fontsize=10)
ax.set_xlabel('F1 delta (no-rules minus full-rules)', fontsize=11)
ax.set_title(
    'F1 delta per KC: positive = no-rules agrees better with humans',
    fontsize=11,
)

from matplotlib.patches import Patch
legend_handles = [
    Patch(color='#EE6677', label='Structural KC'),
    Patch(color='#BBBBBB', label='Other KC'),
]
ax.legend(handles=legend_handles, fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

fig2_path = FIG_DIR / 'fig2_f1_delta_diverging.png'
fig.savefig(fig2_path, dpi=300)
plt.close(fig)
print(f'Saved {fig2_path}')

Saved results\figures\fig2_f1_delta_diverging.png


In [10]:
# Figure 3: Heatmap — structural KCs × 4 sources, cells = tag counts
sources = ['HumA_count', 'HumB_count', 'Full_count', 'NoRules_count']
source_labels = ['Human A', 'Human B', 'Full-rules V3', 'No-rules V3']

heatmap_data = struct_df[sources].values.astype(float)

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd')

ax.set_xticks(np.arange(len(source_labels)))
ax.set_yticks(np.arange(len(STRUCTURAL)))
ax.set_xticklabels(source_labels, fontsize=10)
ax.set_yticklabels(STRUCTURAL, fontsize=10)

for i in range(len(STRUCTURAL)):
    for j in range(len(source_labels)):
        val = int(heatmap_data[i, j])
        text_color = 'white' if heatmap_data[i, j] > heatmap_data.max() * 0.6 else 'black'
        ax.text(j, i, str(val), ha='center', va='center', fontsize=11, color=text_color)

plt.colorbar(im, ax=ax, label='Tag count')
ax.set_title('Structural KC tag counts by source', fontsize=11)
plt.tight_layout()

fig3_path = FIG_DIR / 'fig3_structural_heatmap.png'
fig.savefig(fig3_path, dpi=300)
plt.close(fig)
print(f'Saved {fig3_path}')

Saved results\figures\fig3_structural_heatmap.png
